## 1. Load Data and encode labels

In [97]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

In [98]:
train_df = pd.read_csv("train.csv", encoding="latin1")
test_df  = pd.read_csv("test.csv", encoding="latin1")

# Keep only required columns
train_df = train_df[["text", "sentiment"]]
test_df  = test_df[["text", "sentiment"]]

# Drop missing values
train_df = train_df.dropna()
test_df  = test_df.dropna()

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

# Assuming dataset has columns: "text" and "sentiment"
print(train_df.head())

Train size: (27480, 2)
Test size: (3534, 2)
                                                text sentiment
0                I`d have responded, if I were going   neutral
1      Sooo SAD I will miss you here in San Diego!!!  negative
2                          my boss is bullying me...  negative
3                     what interview! leave me alone  negative
4   Sons of ****, why couldn`t they put them on t...  negative


In [99]:
#Encode labels

label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["sentiment"])
test_df["label"] = label_encoder.transform(test_df["sentiment"])

num_labels = len(label_encoder.classes_)
print("Labels:", label_encoder.classes_)

Labels: ['negative' 'neutral' 'positive']


##3. Stratified sampling for reasonable computation time

In [100]:
from sklearn.model_selection import train_test_split

# Sample 10,000 rows stratified by sentiment, to ensure representation of all sentiments in training
sample_size = 4000

full_sampled, _ = train_test_split(
    train_df,
    train_size=sample_size,
    stratify=train_df["sentiment"],
    random_state=42
)

train_sampled,val_sampled = train_test_split(
    full_sampled,
    test_size=0.2,
    stratify=full_sampled["sentiment"],
    random_state=42
)

print(train_sampled["sentiment"].value_counts())
print(val_sampled["sentiment"].value_counts())

sentiment
neutral     1295
positive     999
negative     906
Name: count, dtype: int64
sentiment
neutral     323
positive    250
negative    227
Name: count, dtype: int64


In [101]:
from sklearn.model_selection import train_test_split

test_sample_size = 1000

test_sampled, _ = train_test_split(
    test_df,
    train_size=test_sample_size,
    stratify=test_df["sentiment"],
    random_state=42
)

print(test_sampled["sentiment"].value_counts())

sentiment
neutral     405
positive    312
negative    283
Name: count, dtype: int64


##3. Fine-tuning the transformer

In [102]:
# Convert to HuggingFace Dataset

train_dataset = Dataset.from_pandas(train_sampled[["text", "label"]])
val_dataset = Dataset.from_pandas(val_sampled[["text", "label"]])
test_dataset = Dataset.from_pandas(test_sampled[["text", "label"]])

In [103]:
# Load Pretrained Model

model_name = "google/electra-small-discriminator"  # small model for faster computation

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [104]:
# Training Arguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train
trainer.train()



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.761369
2,No log,0.669361
3,0.793418,0.655082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=600, training_loss=0.7540017255147298, metrics={'train_runtime': 2218.5608, 'train_samples_per_second': 4.327, 'train_steps_per_second': 0.27, 'total_flos': 70609032806400.0, 'train_loss': 0.7540017255147298, 'epoch': 3.0})

##4. Evaluation and classification report

In [105]:
# Evaluation
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Classification Report:

              precision    recall  f1-score   support

    negative       0.69      0.84      0.75       283
     neutral       0.75      0.64      0.69       405
    positive       0.81      0.80      0.81       312

    accuracy                           0.75      1000
   macro avg       0.75      0.76      0.75      1000
weighted avg       0.75      0.75      0.75      1000

